#Phase 1 — Load the Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/final_uci_drug_review_ADE_sentiment.csv")

print(df.shape)
df.head()

(161297, 15)


,uniqueID,drugName,condition,review,rating,date,usefulCount,ade_entities,ade_count,silver_label,max_confidence,max_severity,max_frequency,sentiment,sentiment_score
0,206461,Valsartan,Left Ventricular Dysfunction,"""It has no side effect, I take it in combinati...",9,2012-05-20,27,[],0,0,0.000,0,0,Neutral,0.646
1,95260,Guanfacine,ADHD,"""My son is halfway through his fourth week of ...",8,2010-04-27,192,['cranky'],1,1,0.806,0,0,Neutral,0.435
2,92703,Lybrel,Birth Control,"""I used to take another oral contraceptive, wh...",5,2009-12-14,17,['brown discharge'],1,1,0.920,0,5,Neutral,0.502
3,138000,Ortho Evra,Birth Control,"""This is my first time using any form of birth...",8,2015-11-03,10,[],0,0,0.000,0,0,Positive,0.847
4,35696,Buprenorphine / naloxone,Opiate Dependence,"""Suboxone has completely turned my life around...",9,2016-11-27,37,['constipation'],1,1,0.950,1,0,Positive,0.899


#Phase 2 — Inspect the Dataset

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 161297 entries, 0 to 161296
Data columns (total 15 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   uniqueID         161297 non-null  int64  
 1   drugName         161297 non-null  object 
 2   condition        159239 non-null  object 
 3   review           161297 non-null  object 
 4   rating           161297 non-null  int64  
 5   date             161297 non-null  object 
 6   usefulCount      161297 non-null  int64  
 7   ade_entities     161297 non-null  object 
 8   ade_count        161297 non-null  int64  
 9   silver_label     161297 non-null  int64  
 10  max_confidence   161297 non-null  float64
 11  max_severity     161297 non-null  int64  
 12  max_frequency    161297 non-null  int64  
 13  sentiment        161297 non-null  object 
 14  sentiment_score  161297 non-null  float64
dtypes: float64(2), int64(7), object(6)
memory usage: 18.5+ MB


In [ ]:
missing = pd.DataFrame({
    "Missing": df.isnull().sum(),
    "Percentage": round(df.isnull().mean()*100,2)
})

missing.sort_values(
    "Missing",
    ascending=False
)

,Missing,Percentage
condition,2058,1.28
uniqueID,0,0.00
drugName,0,0.00
review,0,0.00
rating,0,0.00
date,0,0.00
usefulCount,0,0.00
ade_entities,0,0.00
ade_count,0,0.00
silver_label,0,0.00


In [ ]:
df.dropna(inplace=True)

In [ ]:
print(df.shape)
print(df.isnull().sum())

(159239, 15)
uniqueID           0
drugName           0
condition          0
review             0
rating             0
date               0
usefulCount        0
ade_entities       0
ade_count          0
silver_label       0
max_confidence     0
max_severity       0
max_frequency      0
sentiment          0
sentiment_score    0
dtype: int64


In [ ]:
df["silver_label"].value_counts()

,count
silver_label,
1,110282
0,48957


In [ ]:
df.describe()

,uniqueID,rating,usefulCount,ade_count,silver_label,max_confidence,max_severity,max_frequency,sentiment_score
count,159239.000000,159239.000000,159239.000000,159239.000000,159239.000000,159239.000000,159239.000000,159239.000000,159239.000000
mean,116032.886812,6.997406,28.186989,1.667858,0.692556,0.644849,0.619484,0.147803,0.701941
std,66961.789469,3.272545,36.521902,1.851003,0.461436,0.429560,1.463653,0.777924,0.177051
min,2.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.335000
25%,58237.000000,5.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.542000
50%,115893.000000,8.000000,16.000000,1.000000,1.000000,0.950000,0.000000,0.000000,0.707000
75%,173863.500000,10.000000,37.000000,2.000000,1.000000,0.950000,0.000000,0.000000,0.869000
max,232291.000000,10.000000,1291.000000,31.000000,1.000000,0.971000,5.000000,5.000000,0.991000


#Phase 3 — Feature Engineering

##Step 1 — Encode Sentiment
*Since sentiment is text, convert it to numbers.*

In [ ]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()

df["sentiment_encoded"] = label_encoder.fit_transform(df["sentiment"])
print(dict(zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
)))

{'Negative': np.int64(0), 'Neutral': np.int64(1), 'Positive': np.int64(2)}


##Step 2 — TF-IDF Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1,2),
    min_df=5
)

In [ ]:
X_text = tfidf.fit_transform(df["review"])

print(X_text.shape)

(159239, 5000)


##Step 3 — Define Feature Lists

###1. Hybrid features



In [ ]:
hybrid_features = [
    "rating",
    "max_severity",
    "max_frequency"
]

###2. Sentiment features

In [ ]:
sentiment_features = [
    "sentiment_encoded",
    "sentiment_score"
]

##Step 4 — Define Target

In [ ]:
y = df["silver_label"]

QUICK CHECK

In [ ]:
print("Dataset shape:", df.shape)
print("TF-IDF shape:", X_text.shape)

print("\nHybrid features:")
print(df[hybrid_features].head())

print("\nSentiment features:")
print(df[sentiment_features].head())

print("\nTarget distribution:")
print(y.value_counts())

Dataset shape: (159239, 16)
TF-IDF shape: (159239, 5000)

Hybrid features:
   rating  max_severity  max_frequency
0       9             0              0
1       8             0              0
2       5             0              5
3       8             0              0
4       9             1              0

Sentiment features:
   sentiment_encoded  sentiment_score
0                  1            0.646
1                  1            0.435
2                  1            0.502
3                  2            0.847
4                  2            0.899

Target distribution:
silver_label
1    110282
0     48957
Name: count, dtype: int64


#Phase 4 — Construct the Three Feature Sets

**Feature Set A (Baseline)**
1. TF-IDF (review)

**Feature Set B**
1. TF-IDF
2. Rating
3. Max Severity
4. Max Frequency

**Feature Set C**
1. TF-IDF
2. Rating
3. Max Severity
4. Max Frequency
5. Sentiment Encoded
6. Sentiment Score


In [ ]:
from scipy.sparse import hstack
from scipy.sparse import csr_matrix

##Feature Set A (Baseline)

In [ ]:
X_A = X_text

##Feature Set B (Hybrid NLP Features)

In [ ]:
X_B = hstack([
    X_text,
    csr_matrix(df[hybrid_features].values)
])

##Feature Set C (Hybrid + Sentiment)

In [ ]:
all_features = hybrid_features + sentiment_features
X_C = hstack([
    X_text,
    csr_matrix(df[all_features].values)
])

SHAPE

In [ ]:
print("Feature Set A:", X_A.shape)
print("Feature Set B:", X_B.shape)
print("Feature Set C:", X_C.shape)

Feature Set A: (159239, 5000)
Feature Set B: (159239, 5003)
Feature Set C: (159239, 5005)


#Phase 5 — Train / Validation / Test Split

70% Train
10% Validation
20% Test

We'll do it in two steps using stratified sampling.

##Step 1 — Import

In [ ]:
from sklearn.model_selection import train_test_split

##Step 2 — Split off the Test Set (20%)

### 1. Feature Set A

In [ ]:
XA_trainval, XA_test, y_trainval, y_test = train_test_split(
    X_A,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

###2. Feature Set B

In [ ]:
XB_trainval, XB_test, _, _ = train_test_split(
    X_B,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

###3. Feature Set C

In [ ]:
XC_trainval, XC_test, _, _ = train_test_split(
    X_C,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

##Step 3 — Create Validation Set

Now split the remaining 80%.
We want :
Train = 70%,
Validation = 10%

Since validation should be 10 out of the remaining 80, we use - 10 / 80 = 0.125

### 1. Feature Set A

In [ ]:
XA_train, XA_val, y_train, y_val = train_test_split(
    XA_trainval,
    y_trainval,
    test_size=0.125,
    random_state=42,
    stratify=y_trainval
)

###2. Feature Set B

In [ ]:
XB_train, XB_val, _, _ = train_test_split(
    XB_trainval,
    y_trainval,
    test_size=0.125,
    random_state=42,
    stratify=y_trainval
)

###3. Feature Set C

In [ ]:
XC_train, XC_val, _, _ = train_test_split(
    XC_trainval,
    y_trainval,
    test_size=0.125,
    random_state=42,
    stratify=y_trainval
)

VERIFY

In [ ]:
print("TRAIN :", XA_train.shape)
print("VALID :", XA_val.shape)
print("TEST  :", XA_test.shape)

TRAIN : (111467, 5000)
VALID : (15924, 5000)
TEST  : (31848, 5000)


##Step 5 — Verify Label Distribution

In [ ]:
print("Train")
print(y_train.value_counts(normalize=True))

print("\nValidation")
print(y_val.value_counts(normalize=True))

print("\nTest")
print(y_test.value_counts(normalize=True))

Train
silver_label
1    0.692555
0    0.307445
Name: proportion, dtype: float64

Validation
silver_label
1    0.69254
0    0.30746
Name: proportion, dtype: float64

Test
silver_label
1    0.692571
0    0.307429
Name: proportion, dtype: float64


#Phase 6 — XGBoost Training

In [ ]:
!pip install xgboost -q

In [ ]:
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

Model Creation

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

Applying Train-Validation-Test on the Model

##Phase 6.1 — Feature Set A

###Train

In [ ]:
xgb_model.fit(
    XA_train,
    y_train
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=-1,
              num_parallel_tree=None, ...)

### Validation

In [ ]:
val_predictions = xgb_model.predict(XA_val)
val_probabilities = xgb_model.predict_proba(XA_val)[:,1]

print("Validation Accuracy :", accuracy_score(y_val, val_predictions))
print("Validation Precision:", precision_score(y_val, val_predictions))
print("Validation Recall   :", recall_score(y_val, val_predictions))
print("Validation F1 Score :", f1_score(y_val, val_predictions))
print("Validation ROC AUC  :", roc_auc_score(y_val, val_probabilities))

Validation Accuracy : 0.8181361466968099
Validation Precision: 0.8536267176900331
Validation Recall   : 0.8900072542618789
Validation F1 Score : 0.8714374500577111
Validation ROC AUC  : 0.8801410562033422


###Test

In [ ]:
test_predictions = xgb_model.predict(XA_test)
test_probabilities = xgb_model.predict_proba(XA_test)[:,1]

In [ ]:
accuracy = accuracy_score(y_test, test_predictions)

precision = precision_score(y_test, test_predictions)
recall = recall_score(y_test, test_predictions)
f1 = f1_score(y_test, test_predictions)
roc_auc = roc_auc_score(y_test, test_probabilities)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

Accuracy : 0.8152
Precision: 0.8530
Recall   : 0.8858
F1 Score : 0.8691
ROC-AUC  : 0.8799


###Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    test_predictions
)
print(cm)

[[ 6425  3366]
 [ 2520 19537]]


###Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        test_predictions
    )
)

              precision    recall  f1-score   support

           0       0.72      0.66      0.69      9791
           1       0.85      0.89      0.87     22057

    accuracy                           0.82     31848
   macro avg       0.79      0.77      0.78     31848
weighted avg       0.81      0.82      0.81     31848



##Phase 6.2 — Feature Set B (TF-IDF + Hybrid Features)

### Train

In [ ]:
# ==========================================================
# XGBoost - Feature Set B
# ==========================================================

xgb_model_B = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model_B.fit(
    XB_train,
    y_train
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=-1,
              num_parallel_tree=None, ...)

### Validation

In [ ]:
# Validation Predictions
val_predictions_B = xgb_model_B.predict(XB_val)
val_probabilities_B = xgb_model_B.predict_proba(XB_val)[:, 1]

print("===== Feature Set B : Validation =====")

print(f"Accuracy : {accuracy_score(y_val, val_predictions_B):.4f}")
print(f"Precision: {precision_score(y_val, val_predictions_B):.4f}")
print(f"Recall   : {recall_score(y_val, val_predictions_B):.4f}")
print(f"F1 Score : {f1_score(y_val, val_predictions_B):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_val, val_probabilities_B):.4f}")

===== Feature Set B : Validation =====
Accuracy : 0.8316
Precision: 0.8802
Recall   : 0.8760
F1 Score : 0.8781
ROC-AUC  : 0.9021


###Test

In [ ]:
# Test Predictions

test_predictions_B = xgb_model_B.predict(XB_test)

test_probabilities_B = xgb_model_B.predict_proba(XB_test)[:, 1]

In [ ]:
accuracy_B = accuracy_score(y_test, test_predictions_B)
precision_B = precision_score(y_test, test_predictions_B)
recall_B = recall_score(y_test, test_predictions_B)
f1_B = f1_score(y_test, test_predictions_B)
roc_auc_B = roc_auc_score(y_test, test_probabilities_B)

print("===== Feature Set B : Test =====")

print(f"Accuracy : {accuracy_B:.4f}")
print(f"Precision: {precision_B:.4f}")
print(f"Recall   : {recall_B:.4f}")
print(f"F1 Score : {f1_B:.4f}")
print(f"ROC-AUC  : {roc_auc_B:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, test_predictions_B))

print("\nClassification Report")
print(classification_report(y_test, test_predictions_B))

===== Feature Set B : Test =====
Accuracy : 0.8290
Precision: 0.8777
Recall   : 0.8750
F1 Score : 0.8764
ROC-AUC  : 0.9019

Confusion Matrix
[[ 7103  2688]
 [ 2758 19299]]

Classification Report
              precision    recall  f1-score   support

           0       0.72      0.73      0.72      9791
           1       0.88      0.87      0.88     22057

    accuracy                           0.83     31848
   macro avg       0.80      0.80      0.80     31848
weighted avg       0.83      0.83      0.83     31848



##Phase 6.3 — Feature Set C (TF-IDF + Hybrid + Sentiment)

### Train

In [ ]:
# ==========================================================
# XGBoost - Feature Set C
# ==========================================================

xgb_model_C = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model_C.fit(
    XC_train,
    y_train
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=-1,
              num_parallel_tree=None, ...)

###Validation

In [ ]:
# Validation Predictions
val_predictions_C = xgb_model_C.predict(XC_val)
val_probabilities_C = xgb_model_C.predict_proba(XC_val)[:, 1]

print("===== Feature Set C : Validation =====")

print(f"Accuracy : {accuracy_score(y_val, val_predictions_C):.4f}")
print(f"Precision: {precision_score(y_val, val_predictions_C):.4f}")
print(f"Recall   : {recall_score(y_val, val_predictions_C):.4f}")
print(f"F1 Score : {f1_score(y_val, val_predictions_C):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_val, val_probabilities_C):.4f}")

===== Feature Set C : Validation =====
Accuracy : 0.8293
Precision: 0.8737
Recall   : 0.8808
F1 Score : 0.8773
ROC-AUC  : 0.9010


###Test

In [ ]:
# Test Predictions

test_predictions_C = xgb_model_C.predict(XC_test)
test_probabilities_C = xgb_model_C.predict_proba(XC_test)[:, 1]

accuracy_C = accuracy_score(y_test, test_predictions_C)
precision_C = precision_score(y_test, test_predictions_C)
recall_C = recall_score(y_test, test_predictions_C)
f1_C = f1_score(y_test, test_predictions_C)
roc_auc_C = roc_auc_score(y_test, test_probabilities_C)

print("===== Feature Set C : Test =====")

print(f"Accuracy : {accuracy_C:.4f}")
print(f"Precision: {precision_C:.4f}")
print(f"Recall   : {recall_C:.4f}")
print(f"F1 Score : {f1_C:.4f}")
print(f"ROC-AUC  : {roc_auc_C:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, test_predictions_C))

print("\nClassification Report")
print(classification_report(y_test, test_predictions_C))

===== Feature Set C : Test =====
Accuracy : 0.8287
Precision: 0.8733
Recall   : 0.8804
F1 Score : 0.8768
ROC-AUC  : 0.9011

Confusion Matrix
[[ 6973  2818]
 [ 2639 19418]]

Classification Report
              precision    recall  f1-score   support

           0       0.73      0.71      0.72      9791
           1       0.87      0.88      0.88     22057

    accuracy                           0.83     31848
   macro avg       0.80      0.80      0.80     31848
weighted avg       0.83      0.83      0.83     31848



##Final Comparison Table

In [ ]:
results = pd.DataFrame({
    "Feature Set": [
        "A (TF-IDF)",
        "B (TF-IDF + Hybrid)",
        "C (TF-IDF + Hybrid + Sentiment)"
    ],
    "Accuracy": [
        accuracy,
        accuracy_B,
        accuracy_C
    ],
    "Precision": [
        precision,
        precision_B,
        precision_C
    ],
    "Recall": [
        recall,
        recall_B,
        recall_C
    ],
    "F1 Score": [
        f1,
        f1_B,
        f1_C
    ],
    "ROC-AUC": [
        roc_auc,
        roc_auc_B,
        roc_auc_C
    ]
})

results

,Feature Set,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,A (TF-IDF),0.815185,0.853032,0.885751,0.869084,0.879900
1,B (TF-IDF + Hybrid),0.829000,0.877746,0.874960,0.876351,0.901923
2,C (TF-IDF + Hybrid + Sentiment),0.828655,0.873269,0.880355,0.876798,0.901051


#Phase 7 - CatBoost

In [ ]:
!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:00


In [ ]:
from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

##Phase 7.1 - Feature Set A (TF-IDF)


###Train

In [ ]:
cat_model = CatBoostClassifier(
    iterations=200,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

cat_model.fit(
    XA_train,
    y_train
)

0:	total: 1.5s	remaining: 4m 59s
100:	total: 3m 29s	remaining: 3m 25s
199:	total: 5m 38s	remaining: 0us


CatBoostClassifier(depth=6, eval_metric='AUC', iterations=200, learning_rate=0.1, loss_function='Logloss', random_seed=42, verbose=100)

###Validation

In [ ]:
val_predictions = cat_model.predict(XA_val)
val_probabilities = cat_model.predict_proba(XA_val)[:,1]

print("===== Feature Set A : Validation =====")
print(f"Accuracy : {accuracy_score(y_val,val_predictions):.4f}")
print(f"Precision: {precision_score(y_val,val_predictions):.4f}")
print(f"Recall   : {recall_score(y_val,val_predictions):.4f}")
print(f"F1 Score : {f1_score(y_val,val_predictions):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_val,val_probabilities):.4f}")

===== Feature Set A : Validation =====
Accuracy : 0.8143
Precision: 0.8403
Recall   : 0.9036
F1 Score : 0.8708
ROC-AUC  : 0.8728


###Test

In [ ]:
test_predictions = cat_model.predict(XA_test)
test_probabilities = cat_model.predict_proba(XA_test)[:,1]

In [ ]:
accuracy = accuracy_score(y_test,test_predictions)
precision = precision_score(y_test,test_predictions)
recall = recall_score(y_test,test_predictions)
f1 = f1_score(y_test,test_predictions)
roc_auc = roc_auc_score(y_test,test_probabilities)

print("===== Feature Set A : Test =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print("\nConfusion Matrix")
print(confusion_matrix(y_test,test_predictions))
print("\nClassification Report")
print(classification_report(y_test,test_predictions))

===== Feature Set A : Test =====
Accuracy : 0.8126
Precision: 0.8396
Recall   : 0.9016
F1 Score : 0.8695
ROC-AUC  : 0.8723

Confusion Matrix
[[ 5993  3798]
 [ 2170 19887]]

Classification Report
              precision    recall  f1-score   support

           0       0.73      0.61      0.67      9791
           1       0.84      0.90      0.87     22057

    accuracy                           0.81     31848
   macro avg       0.79      0.76      0.77     31848
weighted avg       0.81      0.81      0.81     31848



##Phase 7.2 - Feature Set B

###Train

In [ ]:
cat_model_B = CatBoostClassifier(
    iterations=200,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

cat_model_B.fit(
    XB_train,
    y_train
)

0:	total: 1.46s	remaining: 4m 50s
100:	total: 2m 30s	remaining: 2m 27s
199:	total: 5m	remaining: 0us


CatBoostClassifier(depth=6, eval_metric='AUC', iterations=200, learning_rate=0.1, loss_function='Logloss', random_seed=42, verbose=100)

###Validation

In [ ]:
val_predictions_B = cat_model_B.predict(XB_val)
val_probabilities_B = cat_model_B.predict_proba(XB_val)[:,1]

print("===== Feature Set B : Validation =====")
print(f"Accuracy : {accuracy_score(y_val,val_predictions_B):.4f}")
print(f"Precision: {precision_score(y_val,val_predictions_B):.4f}")
print(f"Recall   : {recall_score(y_val,val_predictions_B):.4f}")
print(f"F1 Score : {f1_score(y_val,val_predictions_B):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_val,val_probabilities_B):.4f}")

===== Feature Set B : Validation =====
Accuracy : 0.8244
Precision: 0.8651
Recall   : 0.8844
F1 Score : 0.8746
ROC-AUC  : 0.8959


###Test

In [ ]:
test_predictions_B = cat_model_B.predict(XB_test)
test_probabilities_B = cat_model_B.predict_proba(XB_test)[:,1]

In [ ]:
accuracy_B = accuracy_score(y_test,test_predictions_B)
precision_B = precision_score(y_test,test_predictions_B)
recall_B = recall_score(y_test,test_predictions_B)
f1_B = f1_score(y_test,test_predictions_B)
roc_auc_B = roc_auc_score(y_test,test_probabilities_B)

print("===== Feature Set B : Test =====")
print(f"Accuracy : {accuracy_B:.4f}")
print(f"Precision: {precision_B:.4f}")
print(f"Recall   : {recall_B:.4f}")
print(f"F1 Score : {f1_B:.4f}")
print(f"ROC-AUC  : {roc_auc_B:.4f}")
print(confusion_matrix(y_test,test_predictions_B))
print(classification_report(y_test,test_predictions_B))

===== Feature Set B : Test =====
Accuracy : 0.8222
Precision: 0.8626
Recall   : 0.8842
F1 Score : 0.8733
ROC-AUC  : 0.8956
[[ 6685  3106]
 [ 2555 19502]]
              precision    recall  f1-score   support

           0       0.72      0.68      0.70      9791
           1       0.86      0.88      0.87     22057

    accuracy                           0.82     31848
   macro avg       0.79      0.78      0.79     31848
weighted avg       0.82      0.82      0.82     31848



##Phase 7.3 - Feature Set C

###Train

In [ ]:
cat_model_C = CatBoostClassifier(
    iterations=200,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

cat_model_C.fit(
    XC_train,
    y_train
)

0:	total: 1.66s	remaining: 5m 30s
100:	total: 2m 35s	remaining: 2m 32s
199:	total: 4m 45s	remaining: 0us


CatBoostClassifier(depth=6, eval_metric='AUC', iterations=200, learning_rate=0.1, loss_function='Logloss', random_seed=42, verbose=100)

###Validation

In [ ]:
val_predictions_C = cat_model_C.predict(XC_val)
val_probabilities_C = cat_model_C.predict_proba(XC_val)[:,1]

print("===== Feature Set C : Validation =====")
print(f"Accuracy : {accuracy_score(y_val,val_predictions_C):.4f}")
print(f"Precision: {precision_score(y_val,val_predictions_C):.4f}")
print(f"Recall   : {recall_score(y_val,val_predictions_C):.4f}")
print(f"F1 Score : {f1_score(y_val,val_predictions_C):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_val,val_probabilities_C):.4f}")

===== Feature Set C : Validation =====
Accuracy : 0.8279
Precision: 0.8625
Recall   : 0.8939
F1 Score : 0.8779
ROC-AUC  : 0.8953


###Test

In [ ]:
test_predictions_C = cat_model_C.predict(XC_test)
test_probabilities_C = cat_model_C.predict_proba(XC_test)[:,1]

In [ ]:
accuracy_C = accuracy_score(y_test,test_predictions_C)
precision_C = precision_score(y_test,test_predictions_C)
recall_C = recall_score(y_test,test_predictions_C)
f1_C = f1_score(y_test,test_predictions_C)
roc_auc_C = roc_auc_score(y_test,test_probabilities_C)

print("===== Feature Set C : Test =====")
print(f"Accuracy : {accuracy_C:.4f}")
print(f"Precision: {precision_C:.4f}")
print(f"Recall   : {recall_C:.4f}")
print(f"F1 Score : {f1_C:.4f}")
print(f"ROC-AUC  : {roc_auc_C:.4f}")

print(confusion_matrix(y_test,test_predictions_C))
print(classification_report(y_test,test_predictions_C))

===== Feature Set C : Test =====
Accuracy : 0.8233
Precision: 0.8587
Recall   : 0.8916
F1 Score : 0.8748
ROC-AUC  : 0.8953
[[ 6554  3237]
 [ 2390 19667]]
              precision    recall  f1-score   support

           0       0.73      0.67      0.70      9791
           1       0.86      0.89      0.87     22057

    accuracy                           0.82     31848
   macro avg       0.80      0.78      0.79     31848
weighted avg       0.82      0.82      0.82     31848



##Final Comparsion Table

In [ ]:
results = pd.DataFrame({

    "Feature Set":[
        "A (TF-IDF)",
        "B (TF-IDF + Hybrid)",
        "C (TF-IDF + Hybrid + Sentiment)"
    ],

    "Accuracy":[accuracy,accuracy_B,accuracy_C],
    "Precision":[precision,precision_B,precision_C],
    "Recall":[recall,recall_B,recall_C],
    "F1 Score":[f1,f1_B,f1_C],
    "ROC-AUC":[roc_auc,roc_auc_B,roc_auc_C]
})
results

,Feature Set,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,A (TF-IDF),0.812610,0.839645,0.901619,0.869529,0.872312
1,B (TF-IDF + Hybrid),0.822249,0.862615,0.884164,0.873256,0.895599
2,C (TF-IDF + Hybrid + Sentiment),0.823317,0.858671,0.891644,0.874847,0.895264
